In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [2]:
output_dir = Path("../output")
datafiles = sorted(output_dir.glob("*/*/experiment_data.*"))
print(f"Found {len(datafiles)} data files:\n")
for f in datafiles:
    task_dir = f.parent.parent.name
    run_dir = f.parent.name
    size_kb = f.stat().st_size / 1024
    print(f"  {task_dir}/{run_dir}/{f.name}  ({size_kb:.1f} KB)")

Found 4 data files:

  stop_signal_task_(rdoc)/2026-03-01_11-52-40/experiment_data.csv  (183.4 KB)
  stop_signal_task_(stop-it)/2026-03-01_11-52-42/experiment_data.csv  (89.7 KB)
  stroop_color-word_task/2026-03-01_11-52-45/experiment_data.csv  (17.5 KB)
  stroop_color-word_task_(rdoc)/2026-03-01_11-52-47/experiment_data.csv  (99.6 KB)


In [4]:
def detect_format(df: pd.DataFrame) -> str:
    """Detect experiment data format from column names."""
    cols = set(df.columns)
    if "signal" in cols and "SSD" in cols:
        return "stopit"
    if "trial_id" in cols and "SS_trial_type" in cols:
        return "expfactory_stop_signal"
    if "trial_id" in cols and "stim_color" in cols:
        return "expfactory_stroop"
    if "text" in cols and "colour" in cols:
        return "cognitionrun_stroop"
    if "trial_id" in cols:
        return "expfactory_generic"
    return "unknown"


def analyze_expfactory_stroop(df: pd.DataFrame) -> dict:
    """Analyze ExpFactory RDoC Stroop data."""
    test = df[df["trial_id"] == "test_trial"].copy()
    test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
    test["correct_trial"] = pd.to_numeric(test["correct_trial"], errors="coerce")

    results = {"task": "Stroop (ExpFactory RDoC)", "n_total": len(test), "conditions": {}}
    for cond in sorted(test["condition"].unique()):
        g = test[test["condition"] == cond]
        rt_correct = g[g["correct_trial"] == 1]["rt"].dropna()
        results["conditions"][cond] = {
            "n": len(g),
            "accuracy": g["correct_trial"].mean(),
            "rt_mean": rt_correct.mean() if len(rt_correct) else np.nan,
            "rt_sd": rt_correct.std() if len(rt_correct) else np.nan,
            "rt_median": rt_correct.median() if len(rt_correct) else np.nan,
        }
    cong = test[(test["condition"] == "congruent") & (test["correct_trial"] == 1)]["rt"].dropna()
    incong = test[(test["condition"] == "incongruent") & (test["correct_trial"] == 1)]["rt"].dropna()
    if len(cong) > 0 and len(incong) > 0:
        results["stroop_effect_ms"] = incong.mean() - cong.mean()
    return results


def analyze_expfactory_stop_signal(df: pd.DataFrame) -> dict:
    """Analyze ExpFactory RDoC Stop Signal data."""
    test = df[df["trial_id"] == "test_trial"].copy()
    test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
    test["correct_trial"] = pd.to_numeric(test["correct_trial"], errors="coerce")

    results = {"task": "Stop Signal (ExpFactory RDoC)", "n_total": len(test), "conditions": {}}
    for cond in sorted(test["condition"].unique()):
        g = test[test["condition"] == cond]
        rt_resp = g["rt"].dropna()
        results["conditions"][cond] = {
            "n": len(g),
            "accuracy": g["correct_trial"].mean(),
            "rt_mean": rt_resp.mean() if len(rt_resp) else np.nan,
            "rt_sd": rt_resp.std() if len(rt_resp) else np.nan,
        }
    return results


def analyze_cognitionrun_stroop(df: pd.DataFrame) -> dict:
    """Analyze Cognition.run Stroop data."""
    trials = df[(df["text"].notna()) & (df["text"] != "") & (df["colour"].notna()) & (df["colour"] != "")].copy()
    trials["rt"] = pd.to_numeric(trials["rt"], errors="coerce")
    trials["condition"] = np.where(trials["text"] == trials["colour"], "congruent", "incongruent")
    key_map = {"red": "r", "green": "g", "blue": "b", "yellow": "y"}
    trials["correct_key"] = trials["colour"].map(key_map)
    trials["correct"] = trials["response"] == trials["correct_key"]

    results = {"task": "Stroop (Cognition.run)", "n_total": len(trials), "conditions": {}}
    for cond in ["congruent", "incongruent"]:
        g = trials[trials["condition"] == cond]
        rt_correct = g[g["correct"]]["rt"].dropna()
        results["conditions"][cond] = {
            "n": len(g),
            "accuracy": g["correct"].mean() if len(g) else np.nan,
            "rt_mean": rt_correct.mean() if len(rt_correct) else np.nan,
            "rt_sd": rt_correct.std() if len(rt_correct) else np.nan,
            "rt_median": rt_correct.median() if len(rt_correct) else np.nan,
        }
    cong = trials[(trials["condition"] == "congruent") & trials["correct"]]["rt"].dropna()
    incong = trials[(trials["condition"] == "incongruent") & trials["correct"]]["rt"].dropna()
    if len(cong) > 0 and len(incong) > 0:
        results["stroop_effect_ms"] = incong.mean() - cong.mean()
    return results


def analyze_stopit(df: pd.DataFrame) -> dict:
    """Analyze STOP-IT Stop Signal data."""
    df["rt"] = pd.to_numeric(df["rt"], errors="coerce")
    df["correct"] = df["correct"].astype(str).str.lower() == "true"
    df["block_i"] = pd.to_numeric(df["block_i"], errors="coerce")
    df["SSD"] = pd.to_numeric(df["SSD"], errors="coerce")

    exp = df[df["block_i"] > 0]  # exclude practice block 0
    go = exp[exp["signal"] == "no"]
    stop = exp[exp["signal"] == "yes"]

    go_correct_rt = go[go["correct"]]["rt"].dropna()
    stop_respond_rt = stop[~stop["correct"]]["rt"].dropna()

    results = {
        "task": "Stop Signal (STOP-IT)",
        "n_total": len(exp),
        "n_practice": len(df[df["block_i"] == 0]),
        "conditions": {
            "go": {
                "n": len(go),
                "accuracy": go["correct"].mean(),
                "rt_mean": go_correct_rt.mean() if len(go_correct_rt) else np.nan,
                "rt_sd": go_correct_rt.std() if len(go_correct_rt) else np.nan,
                "omission_pct": (go["rt"].isna() | (go["response"] == "undefined")).mean() * 100,
            },
            "stop": {
                "n": len(stop),
                "accuracy": stop["correct"].mean(),
                "n_successful": int(stop["correct"].sum()),
                "n_failed": int((~stop["correct"]).sum()),
                "signal_respond_rt": stop_respond_rt.mean() if len(stop_respond_rt) else np.nan,
                "ssd_mean": stop["SSD"].mean(),
                "ssd_range": (stop["SSD"].min(), stop["SSD"].max()),
            },
        },
    }

    # Per-block breakdown
    blocks = {}
    for block in sorted(exp["block_i"].unique()):
        b = exp[exp["block_i"] == block]
        bg = b[b["signal"] == "no"]
        bs = b[b["signal"] == "yes"]
        blocks[int(block)] = {
            "go_acc": bg["correct"].mean() if len(bg) else np.nan,
            "stop_acc": bs["correct"].mean() if len(bs) else np.nan,
            "go_rt": bg[bg["correct"]]["rt"].dropna().mean() if len(bg[bg["correct"]]) else np.nan,
        }
    results["blocks"] = blocks
    return results


ANALYZERS = {
    "expfactory_stroop": analyze_expfactory_stroop,
    "expfactory_stop_signal": analyze_expfactory_stop_signal,
    "cognitionrun_stroop": analyze_cognitionrun_stroop,
    "stopit": analyze_stopit,
}

print("Defined analyzers for:", list(ANALYZERS.keys()))

Defined analyzers for: ['expfactory_stroop', 'expfactory_stop_signal', 'cognitionrun_stroop', 'stopit']


In [5]:
def print_results(r: dict):
    """Pretty-print analysis results."""
    print(f"\n{'=' * 60}")
    print(f"  {r['task']}")
    print(f"{'=' * 60}")
    print(f"  Total test trials: {r['n_total']}")

    if "stroop_effect_ms" in r:
        print(f"  Stroop effect: {r['stroop_effect_ms']:.0f}ms")

    if "n_practice" in r:
        print(f"  Practice trials: {r['n_practice']}")

    print()
    for cond, stats in r["conditions"].items():
        print(f"  {cond}:")
        print(f"    N = {stats['n']}")
        if "accuracy" in stats and not np.isnan(stats["accuracy"]):
            n_correct = int(stats["accuracy"] * stats["n"])
            print(f"    Accuracy: {stats['accuracy']:.1%} ({n_correct}/{stats['n']})")
        if "rt_mean" in stats and not np.isnan(stats.get("rt_mean", np.nan)):
            rt_str = f"    RT: {stats['rt_mean']:.0f}ms"
            if "rt_sd" in stats and not np.isnan(stats.get("rt_sd", np.nan)):
                rt_str += f" (sd={stats['rt_sd']:.0f})"
            if "rt_median" in stats and not np.isnan(stats.get("rt_median", np.nan)):
                rt_str += f" median={stats['rt_median']:.0f}"
            print(rt_str)
        if "omission_pct" in stats:
            print(f"    Omissions: {stats['omission_pct']:.1f}%")
        if "n_successful" in stats:
            print(f"    Successful inhibitions: {stats['n_successful']}")
            print(f"    Signal-respond (failed): {stats['n_failed']}")
        if "signal_respond_rt" in stats and not np.isnan(stats.get("signal_respond_rt", np.nan)):
            print(f"    Signal-respond RT: {stats['signal_respond_rt']:.0f}ms")
        if "ssd_mean" in stats and not np.isnan(stats.get("ssd_mean", np.nan)):
            lo, hi = stats["ssd_range"]
            print(f"    SSD: mean={stats['ssd_mean']:.0f}ms, range=[{lo:.0f}, {hi:.0f}]ms")
        print()

    if "blocks" in r:
        print("  Per-block breakdown:")
        for block, bstats in r["blocks"].items():
            go_rt = f"{bstats['go_rt']:.0f}ms" if not np.isnan(bstats["go_rt"]) else "N/A"
            print(f"    Block {block}: go_acc={bstats['go_acc']:.0%}, stop_acc={bstats['stop_acc']:.0%}, go_rt={go_rt}")


# Analyze all data files
all_results = []
for f in datafiles:
    df = pd.read_csv(f)
    fmt = detect_format(df)
    label = f"{f.parent.parent.name}/{f.parent.name}"

    if fmt in ANALYZERS:
        results = ANALYZERS[fmt](df)
        results["file"] = label
        results["format"] = fmt
        all_results.append(results)
        print_results(results)
    else:
        print(f"\n--- {label} ---")
        print(f"  Format: {fmt} (no analyzer)")
        print(f"  Rows: {len(df)}, Columns: {list(df.columns)[:8]}...")


  Stop Signal (ExpFactory RDoC)
  Total test trials: 180

  go:
    N = 120
    Accuracy: 90.8% (109/120)
    RT: 636ms (sd=106)

  stop:
    N = 60
    Accuracy: 46.7% (28/60)
    RT: 630ms (sd=109)


  Stop Signal (STOP-IT)
  Total test trials: 256
  Practice trials: 32

  go:
    N = 192
    Accuracy: 93.8% (180/192)
    RT: 450ms (sd=98)
    Omissions: 6.2%

  stop:
    N = 64
    Accuracy: 7.8% (5/64)
    Successful inhibitions: 5
    Signal-respond (failed): 59
    Signal-respond RT: 448ms
    SSD: mean=107ms, range=[50, 600]ms

  Per-block breakdown:
    Block 1: go_acc=92%, stop_acc=0%, go_rt=435ms
    Block 2: go_acc=96%, stop_acc=0%, go_rt=456ms
    Block 3: go_acc=96%, stop_acc=19%, go_rt=447ms
    Block 4: go_acc=92%, stop_acc=12%, go_rt=460ms

  Stroop (Cognition.run)
  Total test trials: 15
  Stroop effect: 139ms

  congruent:
    N = 8
    Accuracy: 100.0% (8/8)
    RT: 450ms (sd=124) median=422

  incongruent:
    N = 7
    Accuracy: 85.7% (6/7)
    RT: 588ms (sd=139) 

In [6]:
# Summary table across all runs
rows = []
for r in all_results:
    for cond, stats in r["conditions"].items():
        rows.append({
            "Task": r["task"],
            "Run": r["file"].split("/")[-1],
            "Condition": cond,
            "N": stats["n"],
            "Accuracy": f"{stats['accuracy']:.1%}" if not np.isnan(stats.get("accuracy", np.nan)) else "",
            "RT (ms)": f"{stats['rt_mean']:.0f}" if not np.isnan(stats.get("rt_mean", np.nan)) else "",
        })

if rows:
    summary = pd.DataFrame(rows)
    print("\nSummary Table\n")
    print(summary.to_string(index=False))


Summary Table

                         Task                 Run   Condition   N Accuracy RT (ms)
Stop Signal (ExpFactory RDoC) 2026-03-01_11-52-40          go 120    90.8%     636
Stop Signal (ExpFactory RDoC) 2026-03-01_11-52-40        stop  60    46.7%     630
        Stop Signal (STOP-IT) 2026-03-01_11-52-42          go 192    93.8%     450
        Stop Signal (STOP-IT) 2026-03-01_11-52-42        stop  64     7.8%        
       Stroop (Cognition.run) 2026-03-01_11-52-45   congruent   8   100.0%     450
       Stroop (Cognition.run) 2026-03-01_11-52-45 incongruent   7    85.7%     588
     Stroop (ExpFactory RDoC) 2026-03-01_11-52-47   congruent  60    91.7%     573
     Stroop (ExpFactory RDoC) 2026-03-01_11-52-47 incongruent  60    86.7%     649
